# Q1 - Verified RAW-FABRID data and train-only EDA
The lecturer notebooks inspire loading, visualisation and tensor inspection. All assessed results use RAW-FABRID, never MNIST/CIFAR. This notebook reads the immutable full study and verifies the included authentic sample without requiring the 4 GB download.

In [ ]:
from pathlib import Path
import sys, json
PROJECT = Path.cwd()
if PROJECT.name == 'notebooks': PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT / 'src'))
from fabric_inspector.common import read_json
from IPython.display import display, Image as DisplayImage, Markdown
import pandas as pd
def read(path): return read_json(PROJECT / path)
def figure(path): display(DisplayImage(filename=str(PROJECT / path)))


In [ ]:
from fabric_inspector.data.manifest import REAL
full = read('data/manifests/split.json')
assert full['provenance'] == REAL
frame = pd.read_csv(PROJECT/'data/manifests/split.csv')
assert len(frame)==19852
display(frame.groupby('split').agg(patches=('patch_id','size'), defective=('label','sum'), sources=('source_id','nunique')))
print(full['split_sha256'])

In [ ]:
for key in ['source_id','group_id','verified_extra_group']:
    assert frame.groupby(key)['split'].nunique().max()==1
print('Source, duplicate-component and publisher-origin separation passed')
display(read('data/manifests/data-audit.json'))

In [ ]:
from fabric_inspector.data.manifest import load_manifest
sample, rows, root = load_manifest(PROJECT/'data/sample-manifests/split.json', verify_files=True)
assert len(rows)==168
print(sample['provenance'], len(rows), 'authentic engineering images; no sample performance claim')

In [ ]:
display(read('docs/eda-statistics.json'))
figure('reports/figures/eda-summary.png')
figure('reports/figures/eda-training-examples.png')
display(read('docs/eda-modelling-findings.json'))

Three constraints: low defect prevalence requires recall-sensitive F2 and weighting; tiny regions require unchanged 256x256 detail; four training origin groups require modest capacity and group isolation. Exact-white mask foreground matches all 687 CSV areas. Origin IDs do not establish physical roll/manufacturer identities.